# Sparse Autoencoders: Discovering Interpretable Features

**Sparse Autoencoders (SAEs)** decompose a model's activations into a sparse dictionary of interpretable features. Instead of looking at individual neurons (which are often **polysemantic** — encoding multiple unrelated concepts), SAEs find **monosemantic** features that correspond to single concepts.

This is the technique behind Anthropic's "Towards Monosemanticity" (2023) and related work.

This notebook covers:
1. Why we need SAEs (the superposition problem)
2. How SAEs work
3. Training an SAE on GPT-2 activations
4. Analyzing discovered features
5. Feature attribution through the unembedding

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt

import irtk
from irtk.sae import (
    SparseAutoencoder, train_sae,
    feature_activation_stats, top_activating_examples,
    feature_logit_attribution,
)
from irtk.datasets import to_tokens

print("Loading GPT-2 Small...")
model = irtk.HookedTransformer.from_pretrained("gpt2")
tokenizer = model.tokenizer
print(f"Loaded: {model.cfg.n_layers} layers, d_model={model.cfg.d_model}")

## 1. The Superposition Problem

Neural networks learn more concepts than they have neurons. This means individual neurons are **polysemantic** — they activate for multiple unrelated concepts.

For example, a single neuron might fire for both "sports" and "cooking" contexts. This makes it hard to understand what the model is computing.

**SAEs solve this** by learning an overcomplete basis (more features than dimensions) that decomposes polysemantic neurons into monosemantic features.

## 2. How SAEs Work

The architecture:
```
Input:    x ∈ R^d_model
Centered: x_c = x - b_dec
Encoded:  f = ReLU(x_c @ W_enc + b_enc)    # f ∈ R^n_features, sparse!
Decoded:  x_hat = f @ W_dec + b_dec         # x_hat ∈ R^d_model
```

The loss: `MSE(x, x_hat) + λ * L1(f)`

- The MSE term ensures good reconstruction
- The L1 term encourages sparse feature activations
- `n_features >> d_model` (overcomplete = more features than dimensions)

## 3. Collecting Activations

First, we need a dataset of activations to train on. We'll collect residual stream activations from a specific layer.

In [ ]:
# Diverse text for collecting activations
texts = [
    "The cat sat on the mat and looked at the fish swimming in the bowl",
    "Scientists discovered a new species of deep sea creature near the ocean floor",
    "The stock market crashed dramatically on Monday causing widespread panic among investors",
    "She walked through the beautiful garden admiring the colorful flowers and tall trees",
    "The president gave a long speech about the economy and future plans for growth",
    "After finishing his homework the boy went outside to play basketball with friends",
    "The old library contained thousands of rare books from centuries past and ancient scrolls",
    "Machine learning algorithms are transforming how we approach complex problems in science and engineering",
    "The restaurant served delicious Italian food including fresh pasta and homemade pizza",
    "During the concert the crowd cheered loudly as the band played their biggest hits",
    "The mountain trail was steep and rocky but the view from the top was breathtaking",
    "New research suggests that regular exercise can significantly improve mental health outcomes",
    "The city council voted to approve the new park construction project downtown",
    "Ancient civilizations built remarkable structures that still stand thousands of years later",
    "The software engineer fixed a critical bug that had been causing system crashes",
    "The weather forecast predicted heavy rain and strong winds for the coming weekend",
]

# Collect activations from a middle layer
target_layer = 6  # middle-ish layer of GPT-2 Small
hook_name = f"blocks.{target_layer}.hook_resid_post"

all_activations = []
all_tokens = []
all_positions = []

for text in texts:
    tokens = to_tokens(text, tokenizer=tokenizer)
    _, cache = model.run_with_cache(tokens)
    resid = cache[hook_name]  # [seq_len, d_model]
    
    for pos in range(len(tokens)):
        all_activations.append(np.array(resid[pos]))
        all_tokens.append(int(tokens[pos]))
        all_positions.append(pos)

activations = jnp.array(np.stack(all_activations))
print(f"Collected {activations.shape[0]} activation vectors from layer {target_layer}")
print(f"Shape: {activations.shape}")

## 4. Training the SAE

We'll train a small SAE for demonstration. In practice, you'd use many more features and much more data.

In [ ]:
# Train SAE with 4x expansion factor
n_features = model.cfg.d_model * 4  # 768 * 4 = 3072 features

print(f"Training SAE: d_model={model.cfg.d_model} -> n_features={n_features}")
print(f"Expansion factor: {n_features / model.cfg.d_model:.0f}x")
print()

result = train_sae(
    activations,
    d_model=model.cfg.d_model,
    n_features=n_features,
    l1_coeff=5e-4,
    lr=3e-4,
    epochs=30,
    batch_size=128,
    verbose=True,
)

sae = result.sae

In [ ]:
# Plot training curves
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

axes[0,0].plot(result.train_losses, label='Total')
axes[0,0].plot(result.val_losses, label='Val')
axes[0,0].set_title('Total Loss')
axes[0,0].legend()
axes[0,0].grid(True, alpha=0.3)

axes[0,1].plot(result.recon_losses)
axes[0,1].set_title('Reconstruction Loss (MSE)')
axes[0,1].grid(True, alpha=0.3)

axes[1,0].plot(result.l1_losses)
axes[1,0].set_title('L1 Loss (Sparsity)')
axes[1,0].grid(True, alpha=0.3)

axes[1,1].plot(result.l0_sparsities)
axes[1,1].set_title(f'L0 Sparsity (avg active features out of {n_features})')
axes[1,1].grid(True, alpha=0.3)

for ax in axes.flat:
    ax.set_xlabel('Epoch')

plt.suptitle('SAE Training Curves', fontsize=14)
plt.tight_layout()
plt.show()

print(f"\nFinal L0 sparsity: {result.l0_sparsities[-1]:.1f} / {n_features} features active")
print(f"Sparsity ratio: {result.l0_sparsities[-1] / n_features:.3f}")

## 5. Analyzing Feature Statistics

In [ ]:
stats = feature_activation_stats(sae, activations)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Firing rate distribution
axes[0].hist(stats['firing_rate'], bins=50, edgecolor='black', alpha=0.7)
axes[0].set_xlabel('Firing Rate')
axes[0].set_ylabel('Count')
axes[0].set_title('Feature Firing Rate Distribution')
axes[0].set_yscale('log')

# Mean activation distribution
axes[1].hist(stats['mean_acts'][stats['mean_acts'] > 0], bins=50, edgecolor='black', alpha=0.7)
axes[1].set_xlabel('Mean Activation')
axes[1].set_ylabel('Count')
axes[1].set_title('Mean Activation (active features only)')

# Max activation distribution
axes[2].hist(stats['max_acts'][stats['max_acts'] > 0], bins=50, edgecolor='black', alpha=0.7)
axes[2].set_xlabel('Max Activation')
axes[2].set_ylabel('Count')
axes[2].set_title('Max Activation Distribution')

plt.tight_layout()
plt.show()

n_dead = int(np.sum(stats['firing_rate'] == 0))
n_alive = n_features - n_dead
print(f"Alive features: {n_alive}/{n_features} ({n_alive/n_features:.1%})")
print(f"Dead features: {n_dead}/{n_features} ({n_dead/n_features:.1%})")
print(f"Average L0: {stats['l0_mean']:.1f}")

## 6. Examining Individual Features

Let's look at what individual features respond to by finding their top activating examples.

In [ ]:
# Find the most active features (highest max activation)
top_features = np.argsort(stats['max_acts'])[::-1][:10]

print("Top 10 most active features:\n")
for feat_idx in top_features:
    indices, act_values = top_activating_examples(sae, feat_idx, activations, k=5)
    
    print(f"Feature {feat_idx} (firing rate: {stats['firing_rate'][feat_idx]:.3f}, max: {stats['max_acts'][feat_idx]:.2f}):")
    for j, (ex_idx, act_val) in enumerate(zip(indices, act_values)):
        tok = tokenizer.decode([all_tokens[ex_idx]])
        print(f"  [{act_val:.2f}] token: {tok!r}")
    print()

## 7. Feature Logit Attribution

Each SAE feature is a direction in activation space. We can project it through the unembedding matrix to see which output tokens it promotes or suppresses. This is called the feature's **logit effect**.

In [ ]:
# Look at top features' logit effects
W_U = model.unembed.W_U

for feat_idx in top_features[:5]:
    promoted, suppressed = feature_logit_attribution(sae, W_U, feat_idx, k=10)
    
    promoted_tokens = [tokenizer.decode([int(t)]) for t in promoted]
    suppressed_tokens = [tokenizer.decode([int(t)]) for t in suppressed]
    
    print(f"Feature {feat_idx}:")
    print(f"  Promotes:   {promoted_tokens[:5]}")
    print(f"  Suppresses: {suppressed_tokens[:5]}")
    print()

## 8. Reconstruction Quality

How well does the SAE reconstruct the original activations? Let's check on a specific example.

In [ ]:
# Pick a random activation and reconstruct it
example_idx = 42
x = activations[example_idx]
x_hat, feat_acts = sae(x[None, :])
x_hat = x_hat[0]

recon_error = float(jnp.mean((x - x_hat) ** 2))
cosine_sim = float(
    jnp.dot(x, x_hat) / (jnp.linalg.norm(x) * jnp.linalg.norm(x_hat))
)
n_active = int(jnp.sum(feat_acts[0] > 0))

print(f"Reconstruction of activation {example_idx}:")
print(f"  MSE: {recon_error:.4f}")
print(f"  Cosine similarity: {cosine_sim:.4f}")
print(f"  Active features: {n_active}/{n_features}")

# Visualize original vs reconstructed
fig, ax = plt.subplots(figsize=(12, 3))
dims = range(min(100, model.cfg.d_model))
ax.plot(dims, np.array(x)[:100], label='Original', alpha=0.7)
ax.plot(dims, np.array(x_hat)[:100], label='Reconstructed', alpha=0.7)
ax.set_xlabel('Dimension (first 100)')
ax.set_ylabel('Value')
ax.set_title(f'Original vs SAE Reconstruction (cosine sim = {cosine_sim:.3f})')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Key Takeaways

1. **Superposition** means models store more features than they have dimensions, making individual neurons hard to interpret.

2. **SAEs learn an overcomplete, sparse basis** that decomposes polysemantic representations into monosemantic features.

3. **Key hyperparameters**:
   - `n_features`: Dictionary size (typically 4-64x d_model)
   - `l1_coeff`: Controls sparsity (higher = fewer active features)
   - Balance: too sparse = poor reconstruction, not sparse enough = not interpretable

4. **Feature analysis** tells you what each feature detects:
   - Top activating examples show what triggers the feature
   - Logit attribution shows what output tokens the feature promotes

5. **Dead features** (never fire) are wasted capacity. This is a common training challenge.

6. **Limitations of this demo**: Real SAE training needs much more data (millions of tokens), larger dictionaries (32k-128k features), and careful hyperparameter tuning. This notebook shows the workflow.

### API Reference

```python
from irtk.sae import (
    SparseAutoencoder,           # The SAE model
    train_sae,                   # Full training loop
    feature_activation_stats,    # Per-feature statistics
    top_activating_examples,     # Find examples that activate a feature
    feature_logit_attribution,   # Project features through unembedding
)

# Quick start:
result = train_sae(
    activations,          # [n_samples, d_model]
    d_model=768,
    n_features=3072,      # 4x overcomplete
    l1_coeff=5e-4,        # Sparsity penalty
    epochs=30,
)
sae = result.sae
```